# BCEWithLogitsLoss : plusieurs présences dans le même extrait

Changer explicitement la cible quand voix et percussion peuvent coexister.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#pertes-multi-etiquettes).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Nous changeons maintenant de tâche. Notre premier classifieur devait choisir une classe exclusive ; ici un mélange contient à la fois voix et percussion.

| Tâche | Annotation pour un extrait | Type | Perte |
| --- | --- | --- | --- |
| Classe exclusive | [1] : percussion | long, forme (1,) | CrossEntropyLoss |
| Présences multiples | [[1,1]] : voix et percussion | float32, forme (1,2) | BCEWithLogitsLoss |

Une cible flottante ne veut pas dire que la classe est approximative : les deux positions portent ici des présences indépendantes.

Prenons deux scores nuls, choisis à la main pour une incertitude égale. La sigmoïde de chaque score vaut 0,5. Chaque présence vraie coûte alors −log(0,5).

## Démonstration 1 · Un coût indépendant par présence

Comparer le contrat de cette perte avec celui de la classification exclusive.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

Les deux uns se trouvent sur la même ligne : un extrait et deux présences. Il ne s’agit pas de deux indices de classe.

In [4]:
scores = torch.tensor([[0., 0.]])
presences = torch.tensor([[1., 1.]], dtype=torch.float32)


In [5]:
observer('scores', scores)
observer('presences', presences)

scores forme (1, 2) type torch.float32 requires_grad False
  valeurs : [[0.0, 0.0]]
presences forme (1, 2) type torch.float32 requires_grad False
  valeurs : [[1.0, 1.0]]


### Calculer l’état suivant

BCEWithLogitsLoss applique le calcul binaire stable à chaque étiquette. none conserve ici deux coûts indépendants.

In [6]:
critere = torch.nn.BCEWithLogitsLoss(reduction="none")
par_etiquette = critere(scores, presences)


In [7]:
observer('par_etiquette', par_etiquette)

par_etiquette forme (1, 2) type torch.float32 requires_grad False
  valeurs : [[0.6931471824645996, 0.6931471824645996]]


### Lire les sorties

mean() rassemble les deux présences. Les probabilités n’ont pas à se sommer à un entre étiquettes.

In [8]:
print("presences", presences.tolist(), "type", presences.dtype)
print("probabilites", scores.sigmoid().tolist())
print("pertes", [[round(v, 6) for v in ligne] for ligne in par_etiquette.tolist()])
print("moyenne", round(par_etiquette.mean().item(), 6))

presences [[1.0, 1.0]] type torch.float32
probabilites [[0.5, 0.5]]
pertes [[0.693147, 0.693147]]
moyenne 0.693147


Le contrat des cibles a changé avec la tâche. La règle « toutes les cibles sont long » serait donc fausse.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Un seul extrait contient voix ET percussion. Écrivez sa cible multi-étiquettes flottante dans cet ordre.

In [9]:
exo_logits = torch.tensor([[.2,-.1]])
exo_cible = ...  # Remplacer les points par votre réponse.
if exo_cible is Ellipsis or exo_cible is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.as_tensor(exo_cible).dtype.is_floating_point
    assert torch.equal(torch.as_tensor(exo_cible),torch.ones_like(exo_logits))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Deux présences simultanées donnent deux 1, dans une ligne de forme (1,2).

</details>

### Exercice 2

Ces deux probabilités sigmoïdes doivent-elles nécessairement sommer à un ? Répondez par True ou False.

In [10]:
exo_probabilites = torch.sigmoid(torch.tensor([[2.,2.]]))
exo_somme_un = ...  # Remplacer les points par votre réponse.
if exo_somme_un is Ellipsis or exo_somme_un is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_somme_un is bool(torch.isclose(exo_probabilites.sum(),torch.tensor(1.)).item())
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Chaque présence est traitée séparément. La softmax répond à une autre convention.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_logits = torch.tensor([[.2,-.1]])
exo_cible = torch.tensor([[1.,1.]])
if exo_cible is Ellipsis or exo_cible is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.as_tensor(exo_cible).dtype.is_floating_point
    assert torch.equal(torch.as_tensor(exo_cible),torch.ones_like(exo_logits))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_probabilites = torch.sigmoid(torch.tensor([[2.,2.]]))
exo_somme_un = False
if exo_somme_un is Ellipsis or exo_somme_un is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_somme_un is bool(torch.isclose(exo_probabilites.sum(),torch.tensor(1.)).item())
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [11]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()